# RS-Solve: Tự động trích xuất màu sắc & sinh câu hỏi Q3-LF (Kaggle Worker)
**Đề tài nghiên cứu:** Dự đoán có chọn lọc dựa trên khả năng phân giải cho MLLM ảnh viễn thám (RGSP - IEEE TGRS).

Notebook này chạy **Cách 2: Tự động hóa 100% bằng Computer Vision (HSV + Core-crop)** trên Kaggle:
- Quét các dataset trong `/kaggle/input/` (DOTA, DIOR, VisDrone, xView, iSAID).
- Lọc đối tượng độc lập, cô lập theo tiêu chuẩn Johnson và kiểm soát chất lượng RS-Solve.
- Trích xuất màu chủ đạo với độ tin cậy >= 65%, phân loại vào 6 màu chuẩn.
- Sinh ~400-500 câu hỏi Q3-LF cân bằng màu sắc (song ngữ Việt - Anh).
- Tự động hợp nhất vào bộ benchmark chuẩn hóa 30.332 câu đã có trên GitHub.
- **Thời gian chạy ước tính:** ~2 - 3 phút.

In [ ]:
# 1. Tải mã nguồn mới nhất từ GitHub
import os, sys, shutil, subprocess
from pathlib import Path

repo_dir = Path("/kaggle/working/RGSP-nckh")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone https://github.com/kttt294/RGSP-nckh.git
!pip install -q Pillow
print("Clone và cài đặt môi trường thành công")

In [ ]:
# 2. Kiểm tra các bộ dữ liệu đã được mount trong /kaggle/input
input_dir = Path("/kaggle/input")
if input_dir.is_dir():
    print("Các bộ dữ liệu phát hiện trong /kaggle/input:")
    for sub in sorted(input_dir.iterdir()):
        if sub.is_dir():
            print(f"  [+] {sub.name}")

In [ ]:
# 3. Chạy thuật toán tự động sinh Q3-LF và hợp nhất vào Benchmark
os.chdir("/kaggle/working/RGSP-nckh/rs-solve")

cmd = [
    sys.executable, "-u", "generate_q3_lf_automated.py",
    "--root-dir", "/kaggle/input",
    "--benchmark-dir", "/kaggle/working/RGSP-nckh/kaggle_benchmark_output",
    "--target-per-color", "70",
    "--target-unanswerable", "50",
    "--min-confidence", "0.65",
    "--min-pixels", "25"
]

p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
print(f"\nQuy trình hoàn tất với mã trạng thái: {p.returncode}")

In [ ]:
# 4. Sao chép các tệp kết quả ra /kaggle/working/ để tải về trực tiếp từ Kaggle UI
import shutil
output_dir = Path("/kaggle/working/RGSP-nckh/kaggle_benchmark_output")
target_dir = Path("/kaggle/working")

files_to_copy = [
    "rs_solve_full_benchmark.jsonl",
    "rs_solve_full_benchmark.json",
    "rs_solve_full_benchmark_en.jsonl",
    "rs_solve_full_benchmark_en.json",
    "rs_solve_full_benchmark_build_report.json"
]

for fname in files_to_copy:
    src = output_dir / fname
    dst = target_dir / fname
    if src.exists():
        shutil.copy2(src, dst)
        print(f"  [✓] Đã xuất {fname} ({dst.stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# 5. Kiểm tra thống kê bộ dữ liệu sau khi bổ sung Q3-LF
import json
report_file = Path("/kaggle/working/rs_solve_full_benchmark_build_report.json")
jsonl_vi = Path("/kaggle/working/rs_solve_full_benchmark.jsonl")
jsonl_en = Path("/kaggle/working/rs_solve_full_benchmark_en.jsonl")

if jsonl_vi.exists() and report_file.exists():
    total_vi = sum(1 for _ in jsonl_vi.open(encoding="utf-8"))
    total_en = sum(1 for _ in jsonl_en.open(encoding="utf-8")) if jsonl_en.exists() else 0
    report = json.loads(report_file.read_text(encoding="utf-8"))
    
    print("=" * 80)
    print("TỔNG KẾT BENCHMARK RS-SOLVE HOÀN CHỈNH (ĐỦ 7 LOẠI CÂU HỎI):")
    print(f"Tổng số câu hỏi Tiếng Việt: {total_vi:,} câu")
    print(f"Tổng số câu hỏi Tiếng Anh  : {total_en:,} câu")
    print("\nPhân bổ số lượng theo từng loại câu hỏi:")
    for k, v in report.get("counts_by_kind", {}).items():
        print(f"  - {k:8s}: {v:,} câu")
    if "Q3-LF" in report.get("selected_by_dataset_and_kind", {}):
        print("\nPhân bổ Q3-LF theo dataset:")
        for ds, c in report["selected_by_dataset_and_kind"]["Q3-LF"].items():
            print(f"  - {ds.upper()}: {c} câu")
    print("=" * 80)
